<a href="https://colab.research.google.com/github/dpalma1996/Derian_Palma_Q-Data-Analytics-2026-2027/blob/main/Tasca_S8_01_Anal%C3%ADtica_de_dades_en_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 08 · Analítica de dades en Pandas
**Derian Palma · IT Academy · Data Analytics**

Este notebook desarrolla el nivel 1 y la primera aproximación al enriquecimiento de datos del nivel 2. La exploración inicial identifica los problemas; los ejercicios siguientes justifican y aplican su tratamiento.

**Idiomas y organización.** Los títulos de nivel y ejercicio se mantienen en catalán. Las instrucciones de cada tarea se conservan completas en catalán, como en el enunciado de la Academia, y se acompañan de su traducción al castellano en la misma celda. Las explicaciones, decisiones y comentarios se redactan en castellano. El inglés aparece en la sintaxis de Python, los nombres de métodos de las bibliotecas y algunos términos técnicos o valores originales del dataset. Los nombres de columnas y los identificadores del código se conservan para mantener la correspondencia con los archivos y las celdas existentes.

Cada tarea sigue el orden: enunciado, traducción, explicación previa, código y comentario de resultados cuando corresponde. Los pasos auxiliares de la solución se presentan como explicaciones, sin atribuirlos al enunciado.

**Cómo ejecutar.** En Colab, sube `sprint8_complex.xlsx`, `ciutats_context.csv` y `politica_salarial.json`. Ejecuta las celdas en orden desde el inicio. Todos los imports se encuentran en la primera celda de código.


In [1]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
import unicodedata
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## Nivell 1 - EDA estructurat i neteja bàsica

Antes de explorar el dataset, cargamos el Excel y mostramos sus primeras filas.

In [2]:
df_original = pd.read_excel('/content/sprint8_complex.xlsx')

display(df_original.head())


,Dades de l'enquesta de treballadors 2025,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,Generades aleatòriament per seguretat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
3,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
4,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7


Pandas ha interpretado el título del documento como cabecera y ha asignado nombres como Unnamed a las demás columnas. Los nombres reales de las variables están en la cuarta fila del Excel. Además, la primera columna contiene una numeración exportada que utilizaremos como índice.
Volvemos a cargar el archivo con header=3 e index_col=0 y comprobamos el resultado.

In [3]:
df_original = pd.read_excel(
    '/content/sprint8_complex.xlsx',
    header=3,
    index_col=0
)

display(df_original.head())


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1


La cabecera ya permite leer la tabla y se aprecian caracteres mal codificados, como en el nombre de Càrrec. Conservamos esta primera lectura para documentar el diagnóstico: la corrección de textos se realizará en el ejercicio 2, sin mezclarla con la exploración inicial.


### Exercici 1: Exploració inicial del dataset


**»** Quantes files i columnes té el dataset?

**Traducción:** ¿Cuántas filas y columnas tiene el dataset?


In [4]:
print('Filas y Columnas:', df_original.shape)


Filas y Columnas: (1007, 19)


**»** Quin tipus de dades conté cada columna?

**Traducción:** ¿Qué tipo de datos contiene cada columna?


In [5]:
print('Tipos de datos por columna:')
print(df_original.dtypes)


Tipos de datos por columna:
Nom                            object
Cognoms                        object
DNI                            object
País d'origen                  object
Ciutat                         object
Dia de Naixement              float64
Mes de Naixement              float64
Any de Naixement              float64
Gènere                         object
Salari mensual                 object
Fills                          object
No Fills                       object
Grup Professional              object
CÃ rrec                        object
Nombre_fills                  float64
Te_cotxe                       object
Km_anuals                     float64
Consum_mitja_L_100km          float64
Temperatura_mitjana_ciutat    float64
dtype: object


**»** Hi ha files duplicades?

**Traducción:** ¿Hay filas duplicadas?


In [6]:
print('Duplicados exactos, excluyendo el índice:', df_original.duplicated().sum())


Duplicados exactos, excluyendo el índice: 4


**»** Les dades semblen coherents a primera vista?

**Traducción:** ¿Los datos parecen coherentes a primera vista?


Mostramos las primeras filas, una muestra aleatoria y las últimas filas para revisar la coherencia general de los datos.

In [7]:
display(df_original.head())
display(df_original.sample(10, random_state=8))
display(df_original.tail())


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
433,Anna,SÃ¡nchez,57957055W,Espanya,Murcia,32.0,0.0,1940.0,D,1078â‚¬,True,NaN,Grup A,Administratiu,1.0,True,80000.0,7.4,18.8
643,Nicolas,David,92727823A,FranÃ§a,Marseille,3.0,4.0,2000.0,D,2936 euros,NaN,True,Grup D,Analista,0.0,False,NaN,NaN,NaN
468,Alba,Vidal,70937796C,Espanya,Sevilla,23.0,5.0,1978.0,NaN,797 â‚¬,True,NaN,Grup A,Data Analyst,4.0,NaN,NaN,NaN,15.6
58,Riccardo,Barbieri,93821894X,ItÃ lia,Venezia,21.0,11.0,1994.0,D,1597â‚¬,NaN,True,Grup B,Administratiu,0.0,True,65950.0,NaN,13.8
200,Ben,Bauer MÃ¼ller,57395992K,Alemanya,Hamburg,32.0,13.0,2026.0,D,"1,499 â‚¬",NaN,True,Grup B,TÃ¨cnic IT,0.0,NaN,NaN,NaN,9.4
499,Aina,Casals,86294869K,Espanya,Valladolid,10.0,2.0,1958.0,H,"1,342 â‚¬",NaN,NaN,Grup B,Administratiu,NaN,False,NaN,NaN,11.1
100,Carlos,MuÃ±oz MartÃ­nez,12176998P,Espanya,Alacant,5.0,12.0,1972.0,M,918 â‚¬,NaN,True,Grup A,TÃ¨cnic IT,0.0,NaN,NaN,NaN,NaN
477,Laia,Torres Romero,28484302B,Espanya,Madrid,9.0,1.0,1958.0,M,1.341 â‚¬,NaN,True,Grup A,Cap de projecte,0.0,True,14892.0,NaN,15.0
895,Ilaria,Rizzo,42823207X,ItÃ lia,Torino,16.0,9.0,1998.0,NaN,"2,180 â‚¬",NaN,True,Grup C,Administratiu,NaN,NaN,NaN,NaN,15.0
41,Paul,Richard Laurent,43567615Z,FranÃ§a,Bordeaux,32.0,99.0,2019.0,Dona,"1,967 â‚¬",NaN,True,Grup B,Cap de projecte,NaN,False,NaN,NaN,15.2


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,molts,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN


head() muestra el principio.

sample() permite revisar filas de distintas partes del dataset. random_state=8 hace que siempre aparezca la misma muestra.

tail() muestra el final, que también puede contener registros extraños.

A primera vista, el dataset contiene información personal, laboral y de movilidad de los trabajadores. Sin embargo, se observan valores ausentes, caracteres mal codificados y salarios escritos en diferentes formatos. Por tanto, los datos necesitan una revisión y limpieza antes de realizar el análisis.
También revisamos conjuntamente los indicadores de hijos y de coche con sus medidas asociadas: un nulo no significa necesariamente cero y dos columnas pueden contener información contradictoria. Estas observaciones orientan las validaciones posteriores; todavía no modificamos los datos.


**»** Detectes possibles problemes o inconsistències evidents?

**Traducción:** ¿Detectas posibles problemas o inconsistencias evidentes?


In [8]:
# Revisar cómo están escritas las categorías
print('Género:', df_original['Gènere'].unique())
print('Grupo profesional:', df_original['Grup Professional'].unique())
print('País de origen:', df_original["País d'origen"].unique())


Género: ['Dona' 'H' 'D' 'F' 'Home' 'NC' 'M' nan 'A' '??']
Grupo profesional: ['Grup B' 'Grup C' 'Grup A' 'Grup D' nan 'Z']
País de origen: ['Espanya' 'Noruega' 'Alemanya' 'ItÃ\xa0lia' 'FranÃ§a' nan 'Narnia']


In [9]:
# Revisar una muestra de los salarios tal como están almacenados
display(df_original['Salari mensual'].head(20))


,Salari mensual
0,"1,469 â‚¬"
1,"2,718 â‚¬"
2,1358 euros
3,"1,478 â‚¬"
4,"1,284 â‚¬"
5,"2,069 â‚¬"
6,"2,349 â‚¬"
7,"2,855 â‚¬"
540,"1,338 â‚¬"
9,"1,450 â‚¬"


In [10]:
# Revisar rangos de las columnas numéricas
display(df_original.describe())


,Dia de Naixement,Mes de Naixement,Any de Naixement,Nombre_fills,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
count,1006.000000,1006.000000,1006.000000,709.000000,315.000000,261.000000,936.000000
mean,15.027833,8.556660,2002.229622,1.203103,55951.050794,11.024521,15.379167
std,10.327436,14.298532,143.118077,1.389404,62095.437005,8.775557,7.276425
min,-99.000000,0.000000,1940.000000,0.000000,-50.000000,0.500000,-10.000000
25%,7.000000,3.250000,1969.000000,0.000000,22632.000000,4.700000,12.800000
50%,15.000000,6.000000,1982.000000,1.000000,56135.000000,9.800000,15.500000
75%,24.000000,10.000000,1995.000000,2.000000,80000.000000,13.600000,18.300000
max,32.000000,99.000000,3000.000000,5.000000,1000000.000000,25.000000,42.000000


In [11]:
print('Valores nulos por columna:')
print(df_original.isna().sum())


Valores nulos por columna:
Nom                             1
Cognoms                         1
DNI                             1
País d'origen                   1
Ciutat                          1
Dia de Naixement                1
Mes de Naixement                1
Any de Naixement                1
Gènere                         41
Salari mensual                  1
Fills                         588
No Fills                      422
Grup Professional               2
CÃ rrec                         2
Nombre_fills                  298
Te_cotxe                      349
Km_anuals                     692
Consum_mitja_L_100km          746
Temperatura_mitjana_ciutat     71
dtype: int64


Mostramos ejemplos de contradicciones entre las columnas de hijos y entre la declaración de coche y los datos de movilidad. Esta selección es solo exploratoria: no modifica ni elimina registros.


In [12]:
mostra_fills = df_original['Fills'].eq(True) & df_original['No Fills'].eq(True)
mostra_cotxe = df_original['Te_cotxe'].eq(False) & (
    pd.to_numeric(df_original['Km_anuals'], errors='coerce').gt(0)
    | df_original['Consum_mitja_L_100km'].notna()
)
print('Ejemplos de declaraciones contradictorias sobre hijos:')
display(df_original.loc[mostra_fills, ['Fills', 'No Fills', 'Nombre_fills']].head())
print('Ejemplos de movilidad que requiere revisión:')
display(df_original.loc[mostra_cotxe, ['Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km']].head())


Ejemplos de declaraciones contradictorias sobre hijos:


,Fills,No Fills,Nombre_fills
11,True,True,1.0
375,True,True,0.0
142,True,True,0.0
155,True,True,0.0
165,True,True,1.0


Ejemplos de movilidad que requiere revisión:


,Te_cotxe,Km_anuals,Consum_mitja_L_100km


Se identifican categorías que podrían representar el mismo concepto con distintas etiquetas, como «D», «F» y «Dona», además de problemas de codificación en nombres, columnas y símbolos monetarios. Los salarios están almacenados como texto y presentan distintos formatos, incluyendo cantidades escritas con palabras. También se observan componentes de fechas fuera de rango, kilometrajes negativos y temperaturas extremas que requieren revisión. Hay valores ausentes, una fila completamente vacía y otra con datos aparentemente ficticios, como «XXXX» y «Narnia». Los valores ausentes no deben interpretarse automáticamente como ceros. Estas incidencias se abordarán en los siguientes ejercicios; por ahora, únicamente se identifican, sin modificar los datos.
Las combinaciones de `Fills`, `No Fills` y `Nombre_fills` requieren comprobar que las declaraciones coincidan con el número de hijos. Del mismo modo, contrastaremos `Te_cotxe` con el kilometraje y el consumo. La codificación se corregirá en el ejercicio 2; las categorías, la coherencia familiar y de movilidad y las imputaciones se tratarán en el ejercicio 3.


### Exercici 2: Validacions bàsiques de les dades i decisions d’eliminació

**»** Corregir possibles problemes de codificació i accentuació en els noms de les columnes i en els valors de text, si escau.

**Traducción:** Corregir posibles problemas de codificación y acentuación en los nombres de las columnas y en los valores de texto, si procede.


Retomamos los problemas de codificación observados en las primeras muestras. La corrección se aplica ahora, en la tarea correspondiente del enunciado, sobre una copia de los datos. La función se utiliza tanto en los encabezados como en los valores de texto de las columnas de tipo object; no se limita a una columna concreta. Si la conversión no es válida, conserva el texto.


In [13]:
df = df_original.copy()
df = df.reset_index(drop=True)

def reparar_text(text):
    if not isinstance(text, str):
        return text
    for codificacio in ['latin1', 'cp1252']:
        try:
            text = text.encode(codificacio).decode('utf-8')
            break
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return text.strip()




Aplicamos la función a los nombres de las columnas y después a sus valores de texto. Mostramos los encabezados y una muestra para comprobar la corrección.


In [14]:
df.columns = [reparar_text(columna) for columna in df.columns]
for columna in df.select_dtypes(include='object'):
    df[columna] = df[columna].map(reparar_text)

print('Nombres de las columnas:')
print(df.columns.tolist())
display(df[['Nom', 'Cognoms', "País d'origen", 'Ciutat',
            'Càrrec', 'Salari mensual']].head())



Nombres de las columnas:
['Nom', 'Cognoms', 'DNI', "País d'origen", 'Ciutat', 'Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'Gènere', 'Salari mensual', 'Fills', 'No Fills', 'Grup Professional', 'Càrrec', 'Nombre_fills', 'Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km', 'Temperatura_mitjana_ciutat']


,Nom,Cognoms,País d'origen,Ciutat,Càrrec,Salari mensual
0,Joana,Gil Navarro,Espanya,Valladolid,Cap de projecte,"1,469 €"
1,Marc,Muñoz,Espanya,Alacant,Senior analyst,"2,718 €"
2,Noa,Serra,Espanya,Alacant,Tècnic IT,1358 euros
3,Pol,Gil,Espanya,Sevilla,Data Analyst,"1,478 €"
4,David,Vila,Espanya,Bilbao,Administratiu,"1,284 €"


Después de crear la función, primero se corrigen los encabezados de todas las columnas y después corrige el contenido de las columnas de tipo object, valor por valor. Dentro de ellas, la función modifica únicamente los textos y deja los demás valores como están. La corrección de codificación permite recuperar tildes y símbolos.

**»** Detectar i eliminar files duplicades exactes, indicant quants registres s'han eliminat.

**Traducción:** Detectar y eliminar filas duplicadas exactas, indicando cuántos registros se han eliminado.


Contamos las repeticiones completas y conservamos la primera aparición de cada fila.


In [15]:
duplicats = df.duplicated()
print('Duplicados exactos detectados:', duplicats.sum())
exclosos_duplicats = df.loc[duplicats].copy()
df = df.drop_duplicates()
print('Duplicados restantes:', df.duplicated().sum())


Duplicados exactos detectados: 4
Duplicados restantes: 0


Se detectan y cuentan las filas duplicadas mediante duplicated(). Antes de eliminarlas, se guarda una copia en exclosos_duplicats para poder revisarlas posteriormente. Después, drop_duplicates() elimina las repeticiones y conserva la primera aparición de cada fila. Finalmente, se comprueba que no quedan duplicados.

**»** Comprovar la unicitat del DNI. En cas de detectar DNIs duplicats, hauràs d'analitzar la coherència de la informació associada i descartar o anul·lar els registres que no siguin fiables, justificant clarament la decisió presa.

**Traducción:** Comprobar la unicidad del DNI. Si se detectan DNI duplicados, habrá que analizar la coherencia de la información asociada y descartar o anular los registros que no sean fiables, justificando claramente la decisión tomada.


Después de eliminar duplicados exactos, buscamos todos los registros con un DNI repetido. `keep=False` marca todas sus apariciones, incluida la primera, para comparar la información antes de decidir qué conservar. Los DNI nulos se revisan en la tarea siguiente.


In [16]:
dni_repetit = df['DNI'].notna() & df['DNI'].duplicated(keep=False)
conflictes = df.loc[dni_repetit].sort_values('DNI')
display(conflictes)
print('DNI repetidos distintos:', conflictes['DNI'].nunique())
print('Registros asociados:', len(conflictes))




,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,"1,676 €",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
654,Oriol,Gómez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 €,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.0,1.0,1968.0,H,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,NaN,"2,000 €",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,"1,176 €",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,1663€,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
89,Guillem,Pujol Costa,54013571W,Espanya,A Coruña,17.0,7.0,1983.0,D,1.574 €,True,NaN,Grup B,Project lead,4.0,False,NaN,NaN,14.6
142,Marta,Ruiz,54013571W,Espanya,Palma,24.0,5.0,1982.0,H,"1,276 €",True,True,Grup A,Analista,0.0,False,NaN,NaN,15.7


DNI repetidos distintos: 5
Registros asociados: 16


Aplicamos el criterio de fiabilidad: al no poder determinar la versión correcta, guardamos todos los registros conflictivos aparte y continuamos únicamente con los DNI no repetidos.


In [17]:
exclosos_conflictes = conflictes.copy()
df = df.loc[~dni_repetit].copy()


Se detectan 5 DNI repetidos en 16 registros. Algunos corresponden a nombres y fechas de nacimiento diferentes; en otros coincide la identidad, pero cambian el salario, la ciudad o el grupo profesional.

**Criterio aplicado:** no conservamos ninguna versión de estos 16 registros en el dataset de análisis. Sin una fuente fiable ni fecha de actualización no podemos decidir cuál es correcta. Elegir el salario más alto sesgaría el análisis salarial; elegir la fila con más datos tampoco garantiza que sean verdaderos. Guardamos todos los registros afectados en `exclosos_conflictes` para su revisión. Esta decisión pierde cobertura, pero evita resolver los conflictos de forma arbitraria.

Esto es distinto de los duplicados exactos: allí sí se conserva la primera aparición, porque todas sus variables coinciden. Aquí `df.loc[~dni_repetit]` conserva únicamente los registros cuyo DNI no está repetido.


**»** Identificar els camps clau que no haurien de contenir valors nuls, detectar registres amb informació insuficient i eliminar únicament aquelles files completament buides o amb informació irrecuperable o no substituïble, explicant per què no és possible ni raonable imputar-les.

**Traducción:** Identificar los campos clave que no deberían contener valores nulos, detectar registros con información insuficiente y eliminar únicamente las filas completamente vacías o con información irrecuperable o no sustituible, explicando por qué no es posible ni razonable imputarlas.


Mostramos las filas completamente vacías antes de eliminarlas. Una fila con algunos nulos se conserva si todavía aporta información útil.


In [18]:
buides = df.isna().all(axis=1)
display(df.loc[buides])
print('Filas completamente vacías detectadas:', buides.sum())
exclosos_buits = df.loc[buides].copy()
df = df.dropna(how='all')


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Filas completamente vacías detectadas: 1


El DNI identifica a cada persona en este análisis. Revisamos su formato: ocho dígitos y una letra.

In [19]:
df['DNI'] = df['DNI'].astype('string').str.strip().str.upper()
dni_valid = df['DNI'].str.fullmatch(r'\d{8}[A-Z]', na=False)
display(df.loc[~dni_valid])
print('Registros con identificador no fiable:', (~dni_valid).sum())
exclosos_identificador = df.loc[~dni_valid].copy()
df = df.loc[dni_valid].copy()


,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,molts,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN


Registros con identificador no fiable: 1


El registro con DNI «XXXX» también presenta nombre y apellidos inservibles, país «Narnia» y datos de nacimiento imposibles. Se descarta porque no permite identificar una persona ni recuperar información fiable.


### Exercici 3: Transformacions necessàries i preparació del dataset

Preparamos las variables para analizarlas. En este ejercicio mantenemos todas las filas conservadas y registramos las correcciones e imputaciones.


In [20]:
df_net = df.copy()
auditoria = pd.DataFrame(index=df_net.index)
auditoria['DNI'] = df_net['DNI']


**»** Normalitzar els valors categòrics per evitar ambigüitats (per exemple, valors que representen el mateix concepte però estan escrits de manera diferent).

**Traducción:** Normalizar los valores categóricos para evitar ambigüedades (por ejemplo, valores que representan el mismo concepto pero están escritos de manera diferente).


Primero mostramos los valores distintos de las columnas categóricas para identificar etiquetas que puedan representar el mismo concepto.

In [21]:
columnes_categories = [
    'Gènere',
    "País d'origen",
    'Ciutat',
    'Grup Professional',
    'Càrrec'
]

for columna in columnes_categories:
    print(f'\n{columna}:')
    print(df_net[columna].unique())



Gènere:
['Dona' 'H' 'D' 'F' 'Home' 'NC' 'M' nan 'A']

País d'origen:
['Espanya' 'Noruega' 'Alemanya' 'Itàlia' 'França']

Ciutat:
['Valladolid' 'Alacant' 'Sevilla' 'Bilbao' 'Las Palmas de Gran Canaria'
 'Palma' 'Barcelona' 'Stavanger' 'Zaragoza' 'Stuttgart' 'Tarragona'
 'València' 'Bergen' 'Milano' 'Màlaga' 'Genova' 'Lyon' 'Paris' 'Toulouse'
 'Napoli' 'Venezia' 'A Coruña' 'Girona' 'Oslo' 'Murcia' 'Bordeaux'
 'Trondheim' 'Madrid' 'Berlin' 'Nice' 'Lille' 'Firenze' 'Roma' 'Bologna'
 'Leipzig' 'Düsseldorf' 'München' 'Torino' 'Nantes' 'Marseille' 'Tromsø'
 'Hamburg' 'Frankfurt' 'Köln']

Grup Professional:
['Grup B' 'Grup C' 'Grup A' 'Grup D']

Càrrec:
['Cap de projecte' 'Senior analyst' 'Tècnic IT' 'Data Analyst'
 'Administratiu' 'Responsable de vendes' 'Analista junior' 'Project lead'
 'Admin.' 'Analyst' 'Analista' 'Data analyst' 'Cap de Projecte']


Con este resultado podremos decidir qué etiquetas unificar. Las abreviaturas ambiguas no deben sustituirse sin conocer su significado.

Normalizamos primero el género: unificamos H como Home y D/F como Dona. Conservamos M y A con la indicación «sin documentar», porque no conocemos su significado, y agrupamos NC y los nulos como «No consta». El resultado muestra las categorías que quedan tras la sustitución.


In [22]:
df_net['Gènere'] = df_net['Gènere'].replace({'H': 'Home', 'D': 'Dona', 'F': 'Dona',
    'M': 'M (sin documentar)', 'A': 'A (sin documentar)', 'NC': 'No consta'}).fillna('No consta')

print('Géneros:', df_net['Gènere'].unique())


Géneros: ['Dona' 'Home' 'No consta' 'M (sin documentar)' 'A (sin documentar)']


Después normalizamos los cargos. Unificamos abreviaturas y variantes identificadas, como Admin./Administratiu, Cap de Projecte/Cap de projecte y Data analyst/Data Analyst. Mostramos las categorías finales para comprobar el resultado.


In [23]:

df_net['Càrrec'] = df_net['Càrrec'].replace({'Admin.': 'Administratiu',
    'Cap de Projecte': 'Cap de projecte', 'Project lead': 'Cap de projecte',
    'Data analyst': 'Data Analyst', 'Analyst': 'Analista'})

print('Cargos:', df_net['Càrrec'].unique())


Cargos: ['Cap de projecte' 'Senior analyst' 'Tècnic IT' 'Data Analyst'
 'Administratiu' 'Responsable de vendes' 'Analista junior' 'Analista']


Los cargos equivalentes quedan agrupados bajo una misma etiqueta. Conservamos las diferencias de especialidad y nivel entre Data Analyst, Analista, Analista junior y Senior analyst: no hay evidencia suficiente para considerarlos un mismo cargo.


**»** Revisar la coherència dels valors nuls, assegurant que:

- Les persones que tenen quilometratge anual i consum mitjà associats disposen de cotxe. En els casos en què consti consum mitjà però no quilometratge anual, caldrà decidir com resoldre aquesta incoherència de manera justificada.
- Les persones que tenen un valor en el nombre de fills efectivament tenen fills.

**Traducción:** Revisar la coherencia de los valores nulos, asegurando que:

- Las personas que tienen kilometraje anual y consumo medio asociados disponen de coche. En los casos en que conste consumo medio pero no kilometraje anual, habrá que decidir cómo resolver esta incoherencia de manera justificada.
- Las personas que tienen un valor en el número de hijos efectivamente tienen hijos.


Antes de revisar la movilidad, convertimos el kilometraje y el consumo a valores numéricos. Los valores que no se puedan interpretar quedan nulos.


In [24]:
for columna in ['Km_anuals', 'Consum_mitja_L_100km']:
    df_net[columna] = pd.to_numeric(df_net[columna], errors='coerce')


Creamos una función para convertir los indicadores de verdadero/falso a un tipo booleano que permite conservar los valores desconocidos.


In [25]:
def convertir_boolea(serie):
    return serie.astype('string').str.lower().map({'true': True, 'false': False}).astype('boolean')


Revisamos primero las medidas. Los km negativos o superiores a **200.000** al año quedan nulos por falta de fiabilidad. El límite superior es un umbral de revisión elegido para este ejercicio. Los consumos positivos extremos se conservan y se marcan, porque no conocemos el vehículo.


In [26]:
cotxe = convertir_boolea(df_net['Te_cotxe'])
auditoria['Km_sospitos'] = df_net['Km_anuals'].notna() & ~df_net['Km_anuals'].between(0, 200000)
df_net.loc[auditoria['Km_sospitos'], 'Km_anuals'] = np.nan
df_net.loc[df_net['Consum_mitja_L_100km'].le(0), 'Consum_mitja_L_100km'] = np.nan
auditoria['Consum_extrem'] = df_net['Consum_mitja_L_100km'].lt(2) | df_net['Consum_mitja_L_100km'].gt(20)


Según el enunciado, aportar km positivos o consumo válido es evidencia de disponer de coche. Si existe consumo pero faltan km, conservamos el consumo y dejamos los km desconocidos. Si declara no tener coche y no aporta movilidad, completamos los km con cero; el consumo queda no aplicable.


In [27]:
evidencia = df_net['Km_anuals'].gt(0) | df_net['Consum_mitja_L_100km'].notna()
auditoria['Cotxe_corregit'] = evidencia & ~cotxe.eq(True).fillna(False)
cotxe.loc[evidencia] = True
df_net['Te_cotxe'] = cotxe
df_net.loc[cotxe.eq(False).fillna(False) & df_net['Km_anuals'].isna(), 'Km_anuals'] = 0
auditoria['Consum_sense_km'] = df_net['Consum_mitja_L_100km'].notna() & df_net['Km_anuals'].isna()


print('Indicadores de coche corregidos:', auditoria['Cotxe_corregit'].sum())
print('Consumo disponible sin km:', auditoria['Consum_sense_km'].sum())


Indicadores de coche corregidos: 0
Consumo disponible sin km: 3


Revisamos también los indicadores de hijos y su número. Un número positivo indica que tiene hijos; cero indica que no tiene. Los números negativos o no enteros quedan desconocidos. Registramos las contradicciones antes de unificar la información.


In [28]:
df_net['Nombre_fills'] = pd.to_numeric(df_net['Nombre_fills'], errors='coerce')
fills, no_fills = convertir_boolea(df_net['Fills']), convertir_boolea(df_net['No Fills'])
nombre = df_net['Nombre_fills'].where(df_net['Nombre_fills'].ge(0) & df_net['Nombre_fills'].mod(1).eq(0))
auditoria['Fills_contradictoris'] = ((fills.eq(True) & no_fills.eq(True)) |
    (nombre.eq(0) & fills.eq(True)) | (nombre.gt(0) & no_fills.eq(True))).fillna(False)

print('Contradicciones detectadas:', auditoria['Fills_contradictoris'].sum())


Contradicciones detectadas: 55


Cuando el número de hijos es válido, tendrá prioridad sobre los indicadores contradictorios. Tener un valor informado no implica tener hijos: el valor cero significa que no tiene.


**»** Unificar la informació sobre fills en una única columna, ja que mantenir diverses columnes amb informació complementària no és òptim i pot generar problemes analítics (redundància o multicolinealitat).

**Traducción:** Unificar la información sobre hijos en una única columna, ya que mantener varias columnas con información complementaria no es óptimo y puede generar problemas analíticos (redundancia o multicolinealidad).


Unificamos la información de las columnas Fills (tiene hijos), No Fills (no tiene hijos) y Nombre_fills (cuántos tiene). Damos prioridad al número de hijos cuando es válido.
El proceso tiene dos etapas. Primero, completamos Nombre_fills con cero cuando falta el número, No Fills indica que no tiene hijos y Fills no lo contradice. Después, guardamos en la auditoría si sabemos que la persona tiene hijos: un número positivo indica que sí y cero indica que no. Si declara tener hijos, pero desconocemos cuántos, conservamos ese hecho en la auditoría y dejamos el número como nulo.

In [29]:
sense_fills = nombre.isna() & no_fills.eq(True).fillna(False) & ~fills.eq(True).fillna(False)
nombre.loc[sense_fills] = 0
df_net['Nombre_fills'] = nombre.astype('Int64')



Una vez actualizado el número de hijos, guardamos aparte si sabemos que la persona tiene hijos, aunque no sepamos cuántos. Después eliminamos Fills y No Fills de la tabla de trabajo y mostramos la distribución resultante.


In [30]:
estat = pd.Series(pd.NA, index=df_net.index, dtype='boolean')
estat.loc[nombre.notna()] = nombre.loc[nombre.notna()].gt(0)
estat.loc[nombre.isna() & fills.eq(True).fillna(False) & ~no_fills.eq(True).fillna(False)] = True
auditoria['Te_fills_conegut'] = estat
df_net = df_net.drop(columns=['Fills', 'No Fills'])


print('Número de hijos desconocido:', df_net['Nombre_fills'].isna().sum())
print(df_net['Nombre_fills'].value_counts(dropna=False))


Número de hijos desconocido: 32
Nombre_fills
0       563
1       170
2        79
4        77
3        64
<NA>     32
Name: count, dtype: Int64


En df_net conservamos Nombre_fills como única columna de hijos y eliminamos Fills y No Fills. La tabla auditoria guarda las contradicciones detectadas anteriormente y el indicador Te_fills_conegut, que utilizaremos para el ajuste salarial.
La auditoría contiene información derivada, no una copia de las tres columnas originales. Los datos originales siguen conservados en df_original; aquí modificamos únicamente la copia de trabajo.
Finalmente, mostramos cuántas personas siguen teniendo un número de hijos desconocido y la cantidad de personas con cada número de hijos, incluyendo los nulos mediante value_counts(dropna=False).

**»** Validar la temperatura mitjana. En cas que aquesta informació falti, hauràs d'imputar-la utilitzant un criteri raonable (per ciutat o per un nivell superior justificat).

**Traducción:** Validar la temperatura media. Si falta esta información, habrá que imputarla utilizando un criterio razonable (por ciudad o por un nivel superior justificado).


Primero convertimos Temperatura_mitjana_ciutat a valores numéricos. Si algún valor no puede convertirse, queda como nulo.
Después revisamos las temperaturas disponibles. Para este ejercicio adoptamos el intervalo de −5 a 30 °C como criterio de revisión de temperaturas medias anuales. Marcamos en auditoria['Temperatura_sospitosa'] los valores fuera del intervalo y los sustituimos por nulos en df_net para tratarlos en la imputación. Este límite es un supuesto del análisis, no una regla indicada en el enunciado ni una prueba de que esos valores sean incorrectos.

In [31]:
df_net['Temperatura_mitjana_ciutat'] = pd.to_numeric(
    df_net['Temperatura_mitjana_ciutat'], errors='coerce'
)

temp = df_net['Temperatura_mitjana_ciutat']

auditoria['Temperatura_sospitosa'] = (
    temp.notna() & ~temp.between(-5, 30)
)

df_net.loc[
    auditoria['Temperatura_sospitosa'],
    'Temperatura_mitjana_ciutat'
] = np.nan

A continuación definimos una función para completar los valores nulos mediante medianas. La reutilizaremos después para los salarios.
La función conserva una copia de los valores anteriores a la imputación y prepara dos resultados: la columna completada y una columna que indica de dónde procede cada valor. Para cada agrupación indicada, calcula la mediana y cuenta cuántos valores disponibles la sustentan. Solo completa los nulos que todavía quedan pendientes y cuyos grupos alcanzan el mínimo de observaciones requerido.
Si al terminar las agrupaciones siguen faltando valores, utiliza la mediana global. Todas las medianas se calculan con los datos anteriores a la imputación, sin usar los valores recién completados como referencia. Si tampoco existe una mediana global disponible, el valor permanece nulo.

In [32]:
def imputar_mediana(dades, columna, nivells, minim=1):
    """Calcula cada mediana solo sobre valores observados, no imputados."""
    original = dades[columna].copy()
    resultat = original.copy()
    font = pd.Series('observat', index=dades.index, dtype='string')
    for claus in nivells:
        grups = dades.groupby(claus, dropna=True)[columna]
        mediana = grups.transform('median')
        nombre = grups.transform('count')
        mascara = resultat.isna() & mediana.notna() & nombre.ge(minim)
        resultat.loc[mascara] = mediana.loc[mascara]
        font.loc[mascara] = 'mediana: ' + ' + '.join(claus)
    mascara = resultat.isna()
    resultat.loc[mascara] = original.median()
    font.loc[mascara] = 'mediana global'
    font.loc[resultat.isna()] = 'pendent: sense donants'
    return resultat, font


Aplicamos la función a la temperatura siguiendo este orden: primero la mediana de las personas con la misma ciudad y país; después, la mediana del país; y, como último recurso, la mediana global. Utilizamos la mediana porque es menos sensible a valores extremos que la media.
El resultado actualizado se guarda en df_net['Temperatura_mitjana_ciutat'] y el origen de cada valor en auditoria['Font_temperatura']. En esta llamada basta con un valor disponible para calcular la mediana de un grupo, porque la función tiene minim=1 como valor predeterminado.
Usamos País d'origen como aproximación geográfica, aunque no garantiza el país de residencia. Esta es una limitación del criterio utilizado.

In [33]:
df_net['Temperatura_mitjana_ciutat'], auditoria['Font_temperatura'] = (
    imputar_mediana(
        df_net,
        'Temperatura_mitjana_ciutat',
        [["País d'origen", 'Ciutat'], ["País d'origen"]]
    )
)

display(auditoria['Font_temperatura'].value_counts())

,count
Font_temperatura,
observat,875
mediana: País d'origen + Ciutat,110


La tabla final cuenta cuántas temperaturas se conservaron como observadas y cuántas se completaron con cada criterio. La auditoría permite distinguir los valores informados de los imputados. No se eliminan personas durante este proceso y los datos originales permanecen en df_original.

**»** Revisar i imputar els salaris faltants, aplicant criteris analítics coherents (com ara càrrec, grup professional, ciutat, gènere o franja d'edat).

**Traducción:** Revisar e imputar los salarios faltantes, aplicando criterios analíticos coherentes (como cargo, grupo profesional, ciudad, género o franja de edad).


Primero mostramos los salarios originales sin cifras para identificar las cantidades escritas con palabras y las etiquetas de datos no disponibles antes de convertirlos.


In [34]:
salaris_text = df['Salari mensual'].astype('string')

sense_xifres = (
    salaris_text.notna()
    & ~salaris_text.str.contains(r'\d', na=False)
)

print('Valores de salario sin cifras:')
print(salaris_text.loc[sense_xifres].unique())

Valores de salario sin cifras:
<StringArray>
['mil vuit-cents', 'dos mil', 'n/d', 'tres mil']
Length: 4, dtype: string


La revisión muestra tres cantidades escritas con palabras: «mil vuit-cents», «dos mil» y «tres mil». También aparece «n/d», que indica un dato no disponible.

Creamos un diccionario para convertir esas tres cantidades a números. Para los demás formatos, retiramos espacios y símbolos monetarios e interpretamos los puntos y comas seguidos de tres cifras como separadores de miles, según los formatos observados. Los valores no interpretables o no positivos quedan nulos.

In [35]:
def convertir_salari(valor):
    if pd.isna(valor):
        return np.nan

    text = str(valor).lower().strip()

    paraules = {
        'mil vuit-cents': 1800,
        'dos mil': 2000,
        'tres mil': 3000
    }

    if text in paraules:
        return paraules[text]

    if text == 'n/d':
        return np.nan

    text = (
        text.replace('euros', '')
            .replace('€', '')
            .replace(' ', '')
    )

    # Separadores de miles: 1.469 o 1,469 → 1469.
    if re.fullmatch(r'\d{1,3}([.,]\d{3})+', text):
        text = text.replace(',', '').replace('.', '')

    # Separador decimal, si hay una o dos cifras después.
    elif re.fullmatch(r'\d+[.,]\d{1,2}', text):
        text = text.replace(',', '.')

    numero = pd.to_numeric(text, errors='coerce')

    return numero if pd.notna(numero) and numero > 0 else np.nan

Aplicamos la función a los salarios de df, que conserva los valores anteriores a esta transformación, y guardamos el resultado numérico en df_net.
Antes de imputar, conservamos una copia de los salarios convertidos en la auditoría. Mostramos cuántos quedan nulos y una muestra de los valores numéricos.

In [36]:
df_net['Salari mensual'] = df['Salari mensual'].map(convertir_salari)

auditoria['Salari_abans_imputacio'] = df_net['Salari mensual'].copy()

print(
    'Salarios pendientes de imputar:',
    df_net['Salari mensual'].isna().sum()
)

display(df_net['Salari mensual'].head(20))

Salarios pendientes de imputar: 14


,Salari mensual
0,1469.0
1,2718.0
2,1358.0
3,1478.0
4,1284.0
5,2069.0
6,2349.0
7,2855.0
8,1338.0
9,1450.0


Completamos los salarios faltantes utilizando la función imputar_mediana, definida anteriormente. Primero usamos la mediana de quienes comparten grupo profesional y cargo, siempre que existan al menos cinco salarios observados. Si no hay suficientes datos, utilizamos la mediana del grupo profesional, con el mismo mínimo. Como último recurso, usamos la mediana global.
Elegimos estas variables porque describen la situación laboral. Los salarios válidos se conservan y las medianas se calculan sin reutilizar los valores imputados. La auditoría registra el criterio aplicado a cada persona.

In [37]:
df_net['Salari mensual'], auditoria['Font_salari'] = imputar_mediana(
    df_net,
    'Salari mensual',
    [['Grup Professional', 'Càrrec'], ['Grup Professional']],
    minim=5
)

salaris_imputats = (
    auditoria['Salari_abans_imputacio'].isna()
    & df_net['Salari mensual'].notna()
)

print('Salarios imputados:', salaris_imputats.sum())
print('Salarios todavía nulos:', df_net['Salari mensual'].isna().sum())

display(auditoria['Font_salari'].value_counts())

Salarios imputados: 14
Salarios todavía nulos: 0


,count
Font_salari,
observat,971
mediana: Grup Professional + Càrrec,14


Se han completado 14 salarios. La tabla muestra cuántos valores se conservaron como observados y cuántos se imputaron con cada criterio. Los valores imputados son estimaciones y pueden reducir la dispersión de los salarios, por lo que quedan identificados en la auditoría. No se elimina ningún registro.

### Exercici 4: Primeres anàlisis descriptives

Calculamos resúmenes salariales a partir del dataset preparado y describimos las diferencias entre grupos.


**»** Construir taules resum descriptives de salaris com una taula resum que permeti veure el sou mig, medià, mínim i màxim per Gènere. Ordena la taula en funció del sou mig.

**Traducción:** Construir tablas resumen descriptivas de salarios, como una tabla que permita ver el sueldo medio, mediano, mínimo y máximo por género. Ordena la tabla en función del sueldo medio.


Agrupamos los salarios por género y calculamos la media, la mediana, el mínimo y el máximo. Añadimos el número de trabajadores para conocer el tamaño de cada grupo y ordenamos la tabla de mayor a menor salario medio. Utilizamos los salarios ya preparados, incluidos los imputados.


In [38]:
resum_genere = df_net.groupby('Gènere')['Salari mensual'].agg(
    Treballadors='count', Mitjana='mean', Mediana='median', Minim='min', Maxim='max'
).sort_values('Mitjana', ascending=False)
display(resum_genere.round(2))


,Treballadors,Mitjana,Mediana,Minim,Maxim
Gènere,,,,,
M (sin documentar),34,2130.29,2151.5,821.0,3610.0
A (sin documentar),24,1690.60,1205.5,673.0,3600.0
Home,324,1650.90,1482.0,726.0,3672.0
No consta,62,1493.19,1336.5,724.0,3600.0
Dona,541,1480.73,1342.0,701.0,3564.0


La tabla permite comparar el salario medio y central de cada grupo, junto con sus valores extremos. El número de trabajadores ayuda a contextualizar las diferencias: una media calculada con pocas personas puede ser menos representativa. Los resultados describen diferencias, pero no explican sus causas.


**»** Crear una taula resum amb el salari mig per gènere (files) i país d'origen (columnes). Afegeix-hi les mitjanes als marges de la taula. Aplica format condicional a la taula per veure en un color més intens els valors més elevats.

**Traducción:** Crear una tabla resumen con el salario medio por género (filas) y país de origen (columnas). Añade las medias en los márgenes de la tabla. Aplica formato condicional para mostrar con un color más intenso los valores más elevados.


Creamos una tabla cruzada con el género en las filas y el país de origen en las columnas. Cada celda muestra el salario medio de esa combinación. Añadimos las medias por género, por país y global en los márgenes, y aplicamos una escala de azules para destacar los valores más elevados.


In [39]:
taula_salaris = pd.crosstab(df_net['Gènere'], df_net["País d'origen"],
    values=df_net['Salari mensual'], aggfunc='mean',
    margins=True, margins_name='Total')
display(taula_salaris.style.format('{:,.2f}', na_rep='—')
        .background_gradient(cmap='Blues', axis=None))


País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,Total
Gènere,,,,,,
A (sin documentar),—,"1,774.36","1,104.33",—,—,"1,690.60"
Dona,"1,429.91","1,478.03","1,404.83","1,465.47","1,779.75","1,480.73"
Home,"2,103.82","1,657.26","1,602.67","1,643.61","1,351.94","1,650.90"
M (sin documentar),—,"2,104.93","3,338.00","1,817.00","2,495.50","2,130.29"
No consta,"1,647.75","1,388.27","1,785.20","1,718.64","1,237.00","1,493.19"
Total,"1,602.50","1,565.76","1,507.95","1,558.71","1,641.23","1,565.02"


Los tonos más intensos indican salarios medios más altos, utilizando la misma escala de color para toda la tabla. La fila y la columna «Total» contienen medias calculadas sobre los trabajadores correspondientes, no sumas ni promedios simples de las medias interiores. El guion indica que no hay un resultado disponible para esa combinación; no representa un salario de cero.


## Nivell 2 - Anàlisi amb dades enriquides i estructures múltiples

Iniciamos el enriquecimiento cargando y revisando las fuentes externas, preparando la edad y uniendo el contexto de ciudades. La aplicación de los incrementos salariales queda fuera del alcance de esta entrega.


### Exercici 1: Enriquiment mitjançant datasets externs

**»** Integrar el dataset principal amb:

- El dataset contextual de ciutats.
- La política salarial definida en format JSON.

**Traducción:** Integrar el dataset principal con:

- El dataset contextual de ciudades.
- La política salarial definida en formato JSON.


Primero cargamos y mostramos las dos fuentes externas para conocer su contenido antes de preparar las variables necesarias. El CSV contiene información contextual de las ciudades y el JSON define las reglas de incremento salarial. El CSV se lee con la codificación cp1252.


In [40]:
BASE = Path('/content')
context = pd.read_csv(BASE / 'ciutats_context.csv', encoding='cp1252')
with open(BASE / 'politica_salarial.json', encoding='utf-8') as fitxer:
    politica = json.load(fitxer)

print('Información contextual de las ciudades:')
display(context)
print('Política salarial:')
print(json.dumps(politica, ensure_ascii=False, indent=2))


Información contextual de las ciudades:


,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus
0,Barcelona,Espanya,78,8.5,Mediterrani
1,Madrid,Espanya,75,8.0,Continental
2,València,Espanya,70,7.5,Mediterrani
3,Sevilla,Espanya,65,7.0,Mediterrani
4,Bilbao,Espanya,72,7.8,Oceànic
5,Zaragoza,Espanya,68,7.0,Continental
6,Murcia,Espanya,60,6.5,Mediterrani
7,Palma,Espanya,73,7.2,Mediterrani
8,Girona,Espanya,66,6.8,Mediterrani
9,Tarragona,Espanya,67,6.9,Mediterrani


Política salarial:
{
  "base_increment_by_group": {
    "Grup A": 5.0,
    "Grup B": 3.5,
    "Grup C": 2.0,
    "Grup D": 8.0
  },
  "children_adjustment": {
    "has_children": 1.5,
    "no_children": 0.0
  },
  "age_adjustment": {
    "18-29": 0.0,
    "30-44": 1.0,
    "45-59": 2.0,
    "60+": 1.5
  },
  "cost_of_living_adjustment": {
    "low": 0.0,
    "medium": 1.0,
    "high": 2.0
  },
  "transport_adjustment": {
    "low": 0.5,
    "medium": 0.0,
    "high": -0.5
  },
  "rules": {
    "max_total_increment_percent": 15,
    "min_salary_threshold": 800
  }
}


El CSV se guarda como un DataFrame llamado `context` y el JSON como un diccionario llamado `politica`. `json.dumps()` muestra las reglas de forma legible: `ensure_ascii=False` conserva las tildes e `indent=2` organiza el contenido con sangría. Todavía no hemos unido estas fuentes con los empleados.


Revisamos los valores distintos de país y ciudad del archivo contextual para detectar problemas de escritura, codificación o etiquetas que necesiten normalizarse antes de unir las tablas.


In [41]:
print('Países del archivo contextual:')
print(context['País'].unique())
print('\nCiudades del archivo contextual:')
print(context['Ciutat'].unique())


Países del archivo contextual:
['Espanya' 'França' 'Itàlia' 'Alemanya' 'Noruega' 'Estats Units'
 'Veneçuela']

Ciudades del archivo contextual:
['Barcelona' 'Madrid' 'València' 'Sevilla' 'Bilbao' 'Zaragoza' 'Murcia'
 'Palma' 'Girona' 'Tarragona' 'A Coru\\u00f1a' 'Valladolid' 'Alacant'
 'Las Palmas de Gran Canaria' 'Paris' 'Marseille' 'Lyon' 'Bordeaux'
 'Lille' 'Milano' 'Roma' 'Torino' 'Genova' 'Venezia' 'Napoli' 'Firenze'
 'Berlin' 'Munchen' 'Hamburg' 'Frankfurt' 'Stuttgart' 'Leipzig' 'Oslo'
 'Bergen' 'Valencia']


En la lista de ciudades aparece `A Coru\u00f1a`, con una secuencia que representa la letra «ñ», pero que está almacenada como texto literal. Corregimos este valor a «A Coruña» para facilitar la coincidencia con el dataset de empleados. La `r` delante del texto permite buscar la secuencia literal, evitando que Python la interprete directamente como «ñ».


In [42]:
context['Ciutat'] = context['Ciutat'].replace({
    r'A Coru\u00f1a': 'A Coruña'
})
print('Ciudades después de la corrección:')
print(context['Ciutat'].unique())


Ciudades después de la corrección:
['Barcelona' 'Madrid' 'València' 'Sevilla' 'Bilbao' 'Zaragoza' 'Murcia'
 'Palma' 'Girona' 'Tarragona' 'A Coruña' 'Valladolid' 'Alacant'
 'Las Palmas de Gran Canaria' 'Paris' 'Marseille' 'Lyon' 'Bordeaux'
 'Lille' 'Milano' 'Roma' 'Torino' 'Genova' 'Venezia' 'Napoli' 'Firenze'
 'Berlin' 'Munchen' 'Hamburg' 'Frankfurt' 'Stuttgart' 'Leipzig' 'Oslo'
 'Bergen' 'Valencia']


La política salarial incluye ajustes por franja de edad. Para aplicarlos necesitamos construir la fecha de nacimiento a partir de las columnas de año, mes y día. Primero convertimos esas columnas a números. Después, `pd.to_datetime()` las combina en una fecha. Con `errors='coerce'`, las fechas incompletas o imposibles, como el 31 de febrero, quedan como `NaT`: el valor de Pandas que representa una fecha no disponible.


In [43]:
columnes_naixement = ['Any de Naixement', 'Mes de Naixement', 'Dia de Naixement']
for columna in columnes_naixement:
    df_net[columna] = pd.to_numeric(df_net[columna], errors='coerce')

parts = df_net[columnes_naixement].copy()
parts.columns = ['year', 'month', 'day']
df_net['Data_naixement'] = pd.to_datetime(parts, errors='coerce')

print('Fechas de nacimiento no disponibles:', df_net['Data_naixement'].isna().sum())
display(df_net[['Data_naixement']].head())


Fechas de nacimiento no disponibles: 77


,Data_naixement
0,1958-03-23
1,1960-11-08
2,1961-04-27
3,1985-10-13
4,1965-11-29


Adoptamos el 31/12/2025 como fecha de referencia fija para este ejercicio. Es un supuesto del análisis, no una fecha confirmada de la encuesta. Mantenerla fija permite reproducir los resultados sin que las edades cambien cada día. Calculamos la diferencia entre el año de referencia y el de nacimiento y restamos uno si la persona todavía no ha cumplido años. Al usar el 31 de diciembre, ninguna fecha de nacimiento válida tiene su cumpleaños pendiente ese año; la comprobación permite cambiar la fecha de referencia posteriormente.


In [44]:
DATA_REFERENCIA = pd.Timestamp('2025-12-31')
data = df_net['Data_naixement']
aniversari_pendent = (
    (data.dt.month > DATA_REFERENCIA.month)
    | ((data.dt.month == DATA_REFERENCIA.month)
       & (data.dt.day > DATA_REFERENCIA.day))
)
edat = DATA_REFERENCIA.year - data.dt.year - aniversari_pendent.astype(int)


Conservamos las edades entre 18 y 100 años como criterio de revisión para este ejercicio. Las edades fuera de ese intervalo quedan desconocidas, sin eliminar a la persona. Este intervalo es una decisión del análisis, no una regla del JSON. Guardamos la edad como `Int64`, que admite enteros y nulos, y registramos los casos sin edad disponible en la auditoría. Después retiramos las columnas separadas de año, mes y día, ya representadas en `Data_naixement`.


In [45]:
df_net['Edat'] = edat.where(edat.between(18, 100)).astype('Int64')
auditoria['Data_o_edat_no_valida'] = df_net['Edat'].isna()
df_net = df_net.drop(columns=columnes_naixement)

print('Edades no disponibles:', df_net['Edat'].isna().sum())
display(df_net[['Data_naixement', 'Edat']].head())


Edades no disponibles: 77


,Data_naixement,Edat
0,1958-03-23,67
1,1960-11-08,65
2,1961-04-27,64
3,1985-10-13,40
4,1965-11-29,60


El resultado muestra cuántas personas tienen una edad desconocida y una muestra de las fechas y edades calculadas. Los casos sin edad permanecen en el dataset, pero no podremos determinar su ajuste por edad sin adoptar un criterio adicional. Las fuentes externas ya están cargadas y la edad está preparada para continuar con la integración.


Preparamos claves auxiliares de ciudad y país en minúsculas y sin tildes para comparar las dos tablas sin alterar los nombres originales. Antes de unirlas, mostramos los nombres de ciudad repetidos en el contexto para comprobar si pueden corresponder a varios países.


In [46]:
def clau_text(text):
    if pd.isna(text):
        return pd.NA
    text = reparar_text(str(text)).casefold()
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))


In [47]:
esquerra, dreta = df_net.copy(), context.copy()
esquerra['clau_ciutat'] = esquerra['Ciutat'].map(clau_text)
esquerra['clau_pais'] = esquerra["País d'origen"].map(clau_text)
dreta['clau_ciutat'] = dreta['Ciutat'].map(clau_text)
dreta['clau_pais'] = dreta['País'].map(clau_text)



Comprobamos los nombres de ciudad repetidos después de normalizarlos; mostramos también su país para observar la ambigüedad.


In [48]:
ciutats_repetides = dreta['clau_ciutat'].duplicated(keep=False)
display(dreta.loc[ciutats_repetides, ['Ciutat', 'País', 'clau_ciutat']].sort_values(['clau_ciutat', 'País']))


,Ciutat,País,clau_ciutat
34,Paris,Estats Units,paris
14,Paris,França,paris
2,València,Espanya,valencia
35,Valencia,Veneçuela,valencia


Para distinguir las ciudades homónimas utilizamos ciudad y país conjuntamente. En la tabla de empleados solo disponemos de País d'origen, que no demuestra el país de residencia: usarlo como aproximación es un supuesto de este ejercicio y puede producir coincidencias incorrectas o dejar personas sin contexto. La integración requiere confirmar ese supuesto antes de un uso real.

La unión izquierda (`how='left'`) conserva a todos los empleados. `validate='many_to_one'` exige una sola fila contextual por combinación ciudad-país; cuando no hay coincidencia, las columnas añadidas quedan nulas.


In [49]:
claus = ['clau_ciutat', 'clau_pais']
if dreta.duplicated(claus).any():
    raise ValueError('El contexto contiene claves ciudad-país repetidas: revisar antes de fusionar.')
dreta = dreta.rename(columns={'Ciutat': 'Ciutat_context', 'País': 'Pais_context'})
df_enriquit = esquerra.merge(dreta, on=claus, how='left', validate='many_to_one', indicator=True)
df_enriquit['Context_disponible'] = df_enriquit['_merge'].eq('both')
df_enriquit = df_enriquit.drop(columns=claus + ['_merge'])

print('Filas antes y después:', len(df_net), len(df_enriquit))
print('Personas con contexto:', df_enriquit['Context_disponible'].sum())



Filas antes y después: 985 985
Personas con contexto: 855


Las claves ciudad-país del contexto deben ser únicas. `validate="many_to_one"` evita multiplicar personas accidentalmente durante la fusión. Mostramos las ciudades que no han encontrado contexto.


In [50]:
display(df_enriquit.loc[~df_enriquit['Context_disponible'],
    ["País d'origen", 'Ciutat']].value_counts().to_frame('Persones'))


Persones
País d'origen Ciutat              
Espanya       Màlaga            53
França        Toulouse          15
              Nantes            12
              Nice              11
Itàlia        Bologna           11
Noruega       Stavanger          9
              Trondheim          6
Alemanya      Düsseldorf         5
              Köln               4
Noruega       Tromsø             4

La unión conserva los empleados y permite identificar quiénes disponen de contexto y quiénes no han encontrado coincidencia. La política salarial queda cargada y revisada como diccionario; sus reglas todavía no se han aplicado. La entrega termina en este punto, sin calcular incrementos ni salarios ajustados.
